# **Parametri**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql.types import StringType, StructType, ArrayType

spark = SparkSession.builder.getOrCreate()

# ---------------------------------------------------------------------------
# CELLA PARAMETRI — valori di default, sovrascritti dalla Pipeline
# tramite i "Base parameters" dell'attività Notebook nel ForEach.
# ---------------------------------------------------------------------------
nome_tabella_bronze = "bronze_product_catalog"   # es. @item().DestinationTable
source_folder       = "product_catalog"          # es. @item().SourceFolder

BRONZE_LAKEHOUSE = "LH_Bronze"
SILVER_LAKEHOUSE = "LH_Silver"

nome_tabella_silver = "silver_product_catalog"

print(f"📥 Tabella sorgente (Bronze) : {nome_tabella_bronze}")
print(f"📤 Tabella destinazione (Silver): {nome_tabella_silver}")
print(f"🗂  Source folder: {source_folder}")


StatementMeta(, 8ead7400-435a-440b-89d7-3eadb18fcab0, 4, Finished, Available, Finished, False)

📥 Tabella sorgente (Bronze) : bronze_product_catalog
📤 Tabella destinazione (Silver): silver_product_catalog
🗂  Source folder: product_catalog


# **Cella Path ABFSS**

In [3]:
bronze_info = notebookutils.lakehouse.get(BRONZE_LAKEHOUSE)
bronze_path = bronze_info["properties"]["abfsPath"]

silver_info = notebookutils.lakehouse.get(SILVER_LAKEHOUSE)
silver_path = silver_info["properties"]["abfsPath"]

print(f"📂 Bronze path : {bronze_path}")
print(f"📂 Silver path : {silver_path}")


StatementMeta(, 8ead7400-435a-440b-89d7-3eadb18fcab0, 5, Finished, Available, Finished, False)

📂 Bronze path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/c9c219f7-14cc-47f2-8994-35d08101e937
📂 Silver path : abfss://47f4b971-7b6b-4af9-8001-77745ea917c0@onelake.dfs.fabric.microsoft.com/b4c95460-8615-46fa-9884-21f085344ad0


# **Funzione di pulizia ricorsiva**

In [4]:

def pulisci_espressione(campo_dataType, espressione_colonna):
    if isinstance(campo_dataType, StringType):
        col_pulita = F.trim(espressione_colonna)
        return F.when(col_pulita == "", None).otherwise(col_pulita)

    elif isinstance(campo_dataType, StructType):
        campi_puliti = [
            pulisci_espressione(
                sotto_campo.dataType,
                espressione_colonna.getField(sotto_campo.name)
            ).alias(sotto_campo.name)
            for sotto_campo in campo_dataType.fields
        ]
        return F.struct(*campi_puliti)

    elif isinstance(campo_dataType, ArrayType):
        tipo_elemento = campo_dataType.elementType
        return F.transform(
            espressione_colonna,
            lambda x: pulisci_espressione(tipo_elemento, x)
        )

    else:
        return espressione_colonna

def pulisci_dataframe(df: DataFrame) -> DataFrame:
    espressioni = [
        pulisci_espressione(campo.dataType, F.col(campo.name)).alias(campo.name)
        for campo in df.schema.fields
    ]
    df_pulito = df.select(*espressioni)
    df_pulito = df_pulito.dropna(how="all")
    return df_pulito

StatementMeta(, 8ead7400-435a-440b-89d7-3eadb18fcab0, 6, Finished, Available, Finished, False)

# **Lettura Bronze e Pulizia**

In [ ]:
# ---------------------------------------------------------------------------
# Legge la tabella dal Bronze e applica la pulizia superficiale ricorsiva
# (gestisce anche struct e array annidati come attributes/images).
# ---------------------------------------------------------------------------
print(f"⏳ Lettura da Bronze: {nome_tabella_bronze}")

try:
    df_bronze = spark.read.format("delta").load(f"{bronze_path}/Tables/{nome_tabella_bronze}")
    righe_bronze = df_bronze.count()

    df_silver = pulisci_dataframe(df_bronze)
    righe_silver = df_silver.count()

    print(f"✅ Pulizia completata")
    print(f"├── Righe Bronze : {righe_bronze:,}")
    print(f"├── Righe Silver : {righe_silver:,}")
    print(f"└── Scartate     : {righe_bronze - righe_silver:,}")

except Exception as e:
    print(f"❌ ERRORE in lettura/pulizia di {nome_tabella_bronze}: {e}")
    raise   # ferma l'esecuzione: senza df_silver le celle successive non hanno senso


# **Visualizzazione dataframe espanso per attributi**

In [7]:
# ---------------------------------------------------------------------------
# ESPANSIONE STRUTTURA: trasforma le colonne annidate in colonne piatte
#   - attributes (struct) → colonne separate (color, warranty_months, ecc.)
#   - images (array)      → una riga per ogni URL, con indice di posizione
# Questa è solo una fase di PREVIEW: nessuna scrittura avviene qui.
# ---------------------------------------------------------------------------

# 1. Espande lo struct "attributes" in colonne di primo livello
df_silver_espanso = df_silver.select(
    "*",
    "attributes.*"
).drop("attributes")

# 2. Esplode l'array "images": una riga per ogni URL, con indice di posizione
#    (image_index = 0 → immagine principale, 1,2,... → secondarie)
df_silver_finale = df_silver_espanso.select(
    "*",
    F.posexplode("images").alias("image_index", "image_url")
).drop("images")

# 3. Verifica schema e anteprima visiva
print("📐 Schema finale (dopo espansione attributes + esplosione images):")
df_silver_finale.printSchema()

print(f"\n👀 Anteprima dati — {df_silver_finale.count():,} righe totali "
      f"(vs {df_silver.count():,} righe pre-espansione, per via dell'esplosione images)")

display(df_silver_finale.limit(20))


StatementMeta(, 8ead7400-435a-440b-89d7-3eadb18fcab0, 9, Finished, Available, Finished, False)

📐 Schema finale (dopo espansione attributes + esplosione images):
root
 |-- product_id: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- color: string (nullable = true)
 |-- warranty_months: long (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- image_index: integer (nullable = false)
 |-- image_url: string (nullable = true)


👀 Anteprima dati — 10,000 righe totali (vs 10,000 righe pre-espansione, per via dell'esplosione images)


SynapseWidget(Synapse.DataFrame, f830e133-f767-4af0-a1f4-2015e46b7c7b)

# **Scrittura nel Silver**

In [8]:
# ---------------------------------------------------------------------------
# SCRITTURA FINALE nel Silver Lakehouse (Full Load - overwrite).
# Scrive df_silver_finale, ovvero la versione già espansa e appiattita,
# pronta per essere consumata direttamente da Power BI o dal layer Gold.
# ---------------------------------------------------------------------------
try:
    righe_scritte = df_silver_finale.count()

    (
        df_silver_finale.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(f"{silver_path}/Tables/{nome_tabella_silver}")
    )

    print(f"✅ Scrittura completata: {nome_tabella_silver}")
    print(f"└── Righe scritte: {righe_scritte:,}")

except Exception as e:
    print(f"❌ ERRORE in scrittura su {nome_tabella_silver}: {e}")
    raise


StatementMeta(, 8ead7400-435a-440b-89d7-3eadb18fcab0, 10, Finished, Available, Finished, False)

✅ Scrittura completata: silver_product_catalog
└── Righe scritte: 10,000
